In [ ]:
from datasets import load_dataset

dataset = load_dataset("Mavkif/roman-urdu-msmarco-dataset",
                       data_files={"train": "queries/roman-ur_Arab_queries.train.tsv"},
                       delimiter="\t",
                       column_names=["query_id", "query"])
print(dataset)
print(dataset['train'][0])

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration
import torch

model_name = "Mavkif/roman-urdu-mt5-mmarco"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.eval()
print("Model loaded successfully!")

In [ ]:
from transformers import MT5ForConditionalGeneration

model2 = MT5ForConditionalGeneration.from_pretrained("Mavkif/roman-urdu-mt5-mmarco")
model2.eval()

test_queries = ["corona vaccine safe hai?", "mobile phone sasta kahan milega", "pakistan mein jobs kaise dhundein"]

passages = [dataset['train'][i]['intahi ki wazahat karen'] for i in range(5)]

def get_relevance(query, passage):
    input_text = "Query: " + query + " Document: " + passage + " Relevant:"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        outputs = model2.generate(**inputs, max_new_tokens=2)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=== Experiment Results ===\n")
for query in test_queries:
    print("Query: " + query)
    for i, passage in enumerate(passages[:3]):
        score = get_relevance(query, passage)
        print("  Passage " + str(i+1) + ": " + passage[:50] + " --> " + score)
    print()

In [ ]:
print("=== Matched vs Unmatched Test ===\n")

# Matched pair - same topic
query_matched = "cradet history par cheetal ka kya matlab hai"
passage_matched = "cradet history mein cheetal ka matlab credit score hota hai"

# Unmatched pair - different topic
query_unmatched = "corona vaccine safe hai?"
passage_unmatched = "cradet history par cheetal ka kya matlab hai"

score1 = get_relevance(query_matched, passage_matched)
score2 = get_relevance(query_unmatched, passage_unmatched)

print("Matched Query+Passage: " + score1)
print("Unmatched Query+Passage: " + score2)

In [ ]:
print("=== Extended Relevance Test ===\n")

test_pairs = [
    ("bank transit number kiya hai", "bank transit number ek 9 digit ka code hota hai", "Matched"),
    ("bank transit number kiya hai", "corona vaccine safe hai pakistan mein", "Unmatched"),
    ("mobile phone sasta kahan milega", "olx aur daraz pe saste mobile milte hain", "Matched"),
    ("mobile phone sasta kahan milega", "dimaghi sehat ke liye exercise zaroori hai", "Unmatched"),
    ("pakistan mein jobs kaise dhundein", "rozee.pk aur linkedin pe pakistan ki jobs milti hain", "Matched"),
]

correct = 0
for query, passage, label in test_pairs:
    score = get_relevance(query, passage)
    expected = "yes" if label == "Matched" else "no"
    status = "✓" if score == expected else "✗"
    correct += 1 if score == expected else 0
    print(status + " [" + label + "] " + query[:35] + " --> " + score)

print("\nAccuracy: " + str(correct) + "/5")
